# Model 3 — EfficientNetB0

EfficientNetB0 is the accuracy-oriented transfer-learning candidate. Its compound scaling balances depth, width and resolution, so it provides a useful comparison against the lighter MobileNetV2 while remaining practical to train.

In [1]:
from pathlib import Path
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
import json, math, sys
import numpy as np
import tensorflow as tf

# Allow TensorFlow to grow GPU memory as needed instead of reserving a fixed block.
_available_gpus = tf.config.list_physical_devices('GPU')
for _gpu in _available_gpus:
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        pass
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
SERVICE_DIR = next(p for p in candidates if (p / 'app').is_dir() and (p / 'requirements.txt').exists())
sys.path.insert(0, str(SERVICE_DIR))
from app.config import ARTIFACT_DIR, DATASET_CSV, IMAGE_SIZE, SEED
from app.data import load_manifest, split_manifest
from app.labels import CLASS_NAMES
from app.metrics import calculate_classification_metrics
from app.model import build_transfer_model
USE_IMAGENET_WEIGHTS = True  # Change to False when training without internet.
tf.keras.utils.set_random_seed(SEED)
print('Service:', SERVICE_DIR)
print('Dataset:', DATASET_CSV)

I0000 00:00:1787782247.010402     402 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787782249.660781     402 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Service: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/ai-service
Dataset: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/Dataset/FracAtlas/dataset.csv


W0000 00:00:1787782251.383506     402 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.


In [2]:
frame = load_manifest()
train, validation, test = split_manifest(frame)
print(f'Usable images: {len(frame):,} | train: {len(train):,} | validation: {len(validation):,} | test: {len(test):,}')
display(frame['label'].value_counts().reindex(CLASS_NAMES).rename('count').to_frame())

W0000 00:00:1787782285.119260     402 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
E0000 00:00:1787782327.154762     402 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 414/454
W0000 00:00:1787782327.154894     402 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop window.
E0000 00:00:1787782327.162177     402 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 398/454
W0000 00:00:1787782327.162225     402 local_rendezvous.cc:412] Local rendezvous is aborting with status: INVALID_ARGUMENT: jpeg::Uncompress failed. Invalid JPEG data or crop window.
E0000 00:00:1787782327.168589     402 jpeg_mem.cc:331] Premature end of JPEG data. Stopped at line 446/454
E0000 00:00:1787782327.250343     402 jpeg_mem.cc:331] Premature end of JPEG data

Usable images: 4,024 | train: 3,219 | validation: 402 | test: 403


,count
label,
NO_FRACTURE,3307
ONE_FRACTURE,546
MULTIPLE_FRACTURES,171


In [3]:
BATCH_SIZE = 8
def make_dataset(dataframe, shuffle=False):
    paths = dataframe['path'].to_numpy()
    labels = dataframe['label_index'].to_numpy(dtype=np.int32)
    def load(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(image, IMAGE_SIZE), label
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(dataframe), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.apply(tf.data.experimental.ignore_errors())
    return dataset.batch(BATCH_SIZE).prefetch(1)
train_dataset = make_dataset(train, True).repeat()
validation_dataset = make_dataset(validation).repeat()
test_dataset = make_dataset(test)
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
weights = compute_class_weight('balanced', classes=np.arange(len(CLASS_NAMES)), y=train['label_index'])
class_weights = {i: float(value) for i, value in enumerate(weights)}
print('Class weights:', class_weights)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.
Class weights: {0: 0.4056710775047259, 1: 2.4553775743707096, 2: 7.8321167883211675}


In [4]:
# Recreate the repeated pipelines here so this training cell is safe to rerun.
train_dataset = make_dataset(train, True).repeat()
validation_dataset = make_dataset(validation).repeat()
TRAIN_STEPS = math.ceil(len(train) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(validation) / BATCH_SIZE)
model = build_transfer_model('efficientnetb0', weights='imagenet' if USE_IMAGENET_WEIGHTS else None)
model.summary()
model_path = ARTIFACT_DIR / 'models' / 'efficientnetb0.keras'
model_path.parent.mkdir(parents=True, exist_ok=True)
callbacks = [tf.keras.callbacks.ModelCheckpoint(model_path, monitor='val_accuracy', save_best_only=True), tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True), tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, min_lr=1e-6)]
history = model.fit(train_dataset, validation_data=validation_dataset, epochs=15, steps_per_epoch=TRAIN_STEPS, validation_steps=VALIDATION_STEPS, class_weight=class_weights, callbacks=callbacks, shuffle=False)

Model: "fracatlas_efficientnetb0"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xray (InputLayer)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_zoom (RandomZoom)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 1280)           │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probabilities (Dense)     │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,213,926 (16.07 MB)

 Trainable params: 164,355 (642.01 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

Epoch 1/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 32s 57ms/step - accuracy: 0.5843 - loss: 0.9610 - val_accuracy: 0.6841 - val_loss: 0.7660 - learning_rate: 0.0010
Epoch 2/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 22s 55ms/step - accuracy: 0.6704 - loss: 0.7831 - val_accuracy: 0.7985 - val_loss: 0.4845 - learning_rate: 0.0010
Epoch 3/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 41s 101ms/step - accuracy: 0.6962 - loss: 0.7427 - val_accuracy: 0.8433 - val_loss: 0.4287 - learning_rate: 0.0010
Epoch 4/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 21s 51ms/step - accuracy: 0.7422 - loss: 0.6348 - val_accuracy: 0.7637 - val_loss: 0.5627 - learning_rate: 0.0010
Epoch 5/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 41s 101ms/step - accuracy: 0.7176 - loss: 0.6333 - val_accuracy: 0.7935 - val_loss: 0.5321 - learning_rate: 0.0010
Epoch 6/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 21s 51ms/step - accuracy: 0.7757 - loss: 0.5524 - val_accuracy: 0.8308 - val_loss: 0.4414 - learning_rate: 3.0000e-04
Epoch 7/15
403/403 ━━━━━━━━━━━━━━━━━━━━ 20s 51ms/step - accuracy: 0.77

In [5]:
best_model = tf.keras.models.load_model(model_path)
predicted = best_model.predict(test_dataset, verbose=0).argmax(axis=1)
actual = np.concatenate([labels.numpy() for _, labels in test_dataset], axis=0)
metrics = {'model': 'efficientnetb0', **calculate_classification_metrics(actual, predicted)}
print(classification_report(actual, predicted, target_names=CLASS_NAMES, zero_division=0).replace('macro avg', 'average'))
(ARTIFACT_DIR / 'models' / 'efficientnetb0_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
print('Saved:', model_path)
print(metrics)

/home/rushd/fracturecare-ai-venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


                    precision    recall  f1-score   support

       NO_FRACTURE       0.91      0.90      0.91       331
      ONE_FRACTURE       0.47      0.53      0.50        55
MULTIPLE_FRACTURES       0.44      0.41      0.42        17

          accuracy                           0.83       403
         average       0.61      0.61      0.61       403
      weighted avg       0.83      0.83      0.83       403

Saved: /mnt/c/Users/Rushd/OneDrive - wslqd/Documents/Uni Documents/ICBT/Development Project Final Year/Final Documents/fracturecare-prototype/ai-service/artifacts/models/efficientnetb0.keras
{'model': 'efficientnetb0', 'accuracy': 0.826302729528536, 'balanced_accuracy': 0.6121061333074839, 'macro_precision': 0.6063626964433416, 'macro_recall': 0.6121061333074839, 'macro_f1': 0.6084855749489896, 'fracture_macro_precision': 0.4526209677419355, 'fracture_macro_recall': 0.4695187165775401, 'fracture_macro_f1': 0.45998445998446}
